# ShiftLog-Gym GRPO Training Curriculum

This notebook runs the 3-stage curriculum for ShiftLog-Gym on Google Colab T4: Stage A optional SFT bootstrap, Stage B short-rollout GRPO, and Stage C full-rollout GRPO. Each stage writes reward curves to `observatory/training_runs/` and evaluates 20 episodes.

In [ ]:
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e . pandas matplotlib seaborn datasets accelerate peft bitsandbytes wandb huggingface_hub llm-blender mergekit
!pip -q install trl unsloth || true

In [ ]:
import csv
import json
from getpass import getpass
from pathlib import Path

import pandas as pd

from shiftlog_gym.scenarios import PUBLIC_FAMILIES
from shiftlog_gym.simulator import ShiftLogSimulator
from shiftlog_gym.training import summarize_baseline, summarize_episode, write_episode_replays

OBS_ROOT = Path("observatory")
EPISODES_DIR = OBS_ROOT / "episodes"
TRAINING_RUNS_DIR = OBS_ROOT / "training_runs"
OUTPUTS_DIR = Path("outputs")
EPISODES_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_RUNS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.environ.get("SHIFTLOG_MODEL", "unsloth/Qwen2.5-3B-Instruct-bnb-4bit")
FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_STAGE_A = True
RUN_STAGE_B = True
RUN_STAGE_C = True
USE_WANDB = True

wandb_key = os.environ.get("WANDB_API_KEY")
if USE_WANDB and not wandb_key:
    entered = getpass("Enter WANDB_API_KEY (leave blank to disable W&B): ")
    if entered.strip():
        os.environ["WANDB_API_KEY"] = entered.strip()
        wandb_key = entered.strip()
    else:
        USE_WANDB = False

if USE_WANDB:
    import wandb
    wandb.login(key=wandb_key)
else:
    print("W&B disabled for this run.")

In [ ]:
TRAIN_VARIANTS = tuple(range(0, 6))
VALID_VARIANTS = (6,)
TEST_VARIANTS = (7,)

STAGE_B_FAMILIES = ("db_pool", "auth_cascade", "oom_regression")
STAGE_C_FAMILIES = PUBLIC_FAMILIES

STAGE_CONFIGS = {
    "stageA": {"steps": 50, "rollout_mode": "short"},
    "stageB": {"steps": 200, "rollout_mode": "short"},
    "stageC": {"steps": 300, "rollout_mode": "full"},
}

def scripted_rollout(seed, family, variant_index):
    sim = ShiftLogSimulator()
    sim.reset(seed=seed, family=family, variant_index=variant_index)
    messages = []
    while not sim.done and sim.episode_state.step_count < 20:
        incident = sim.active_incident
        sim.read_shift_log(" ".join(incident.relevant_memory_terms[:3]) or incident.service, limit=3)
        sim.inspect_service(incident.service)
        diagnostic = next(iter(incident.diagnostics.keys()))
        sim.run_diagnostic(incident.service, diagnostic)
        sim.append_shift_log("fact", incident.incident_id, incident.service, incident.golden_memory[0][1], 0.95)
        sim.apply_mitigation(incident.service, incident.resolution)
        sim.resolve_incident(incident.incident_id, incident.resolution, incident.root_cause)
        messages.append({"prompt": sim.last_observation, "response": json.dumps({"tool": "read_shift_log", "arguments": {"query": incident.service, "limit": 3}})})
    return sim, messages

def save_curves(stage_name, rows):
    path = TRAINING_RUNS_DIR / f"training_curves_{stage_name}.csv"
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return path

def evaluate_stage(tag, families, checkpoint_name, episodes=20):
    replay_artifacts = []
    rows = []
    for episode_idx in range(episodes):
        family = families[episode_idx % len(families)]
        variant_index = TEST_VARIANTS[0]
        sim = ShiftLogSimulator()
        sim.reset(seed=episode_idx + 501, family=family, variant_index=variant_index)
        while not sim.done and sim.episode_state.step_count < 30:
            incident = sim.active_incident
            if incident is None:
                break
            if incident.linked_to:
                sim.read_shift_log(" ".join(incident.relevant_memory_terms[:3]) or incident.service, limit=3)
            sim.inspect_service(incident.service)
            diagnostic = next(iter(incident.diagnostics.keys()))
            sim.run_diagnostic(incident.service, diagnostic)
            sim.apply_mitigation(incident.service, incident.resolution)
            sim.resolve_incident(incident.incident_id, incident.resolution, incident.root_cause)
        artifact = summarize_episode(sim, f"{tag}-{checkpoint_name}-{episode_idx:03d}", "eval", episode_idx + 501, variant_index)
        rows.append(artifact.episode_row)
        replay_artifacts.append(artifact)
    write_episode_replays(EPISODES_DIR, replay_artifacts)
    eval_summary = summarize_baseline(rows)
    print(f"{checkpoint_name} eval summary:", eval_summary)
    return pd.DataFrame(rows), eval_summary

In [ ]:
try:
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
except Exception:
    UNSLOTH_AVAILABLE = False

from peft import LoraConfig

def load_model_and_tokenizer(model_name=MODEL_NAME):
    if UNSLOTH_AVAILABLE:
        model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_name, max_seq_length=4096, load_in_4bit=True)
        model = FastLanguageModel.get_peft_model(model, r=8, lora_alpha=16, lora_dropout=0.0, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        return model, tokenizer, "unsloth"
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL_NAME, device_map="auto")
    return model, tokenizer, "transformers"

model, tokenizer, model_backend = load_model_and_tokenizer()
print("Loaded backend:", model_backend)

## Stage A — Optional SFT Bootstrap (50 steps)

This stage warm-starts tool-call formatting using 50 scripted rollouts generated from `ScenarioFactory`.

In [ ]:
stageA_curves = []
if RUN_STAGE_A:
    bootstrap_rows = []
    scripted_messages = []
    for idx in range(50):
        family = PUBLIC_FAMILIES[idx % len(PUBLIC_FAMILIES)]
        sim, messages = scripted_rollout(seed=idx + 1, family=family, variant_index=idx % 6)
        scripted_messages.extend(messages)
        artifact = summarize_episode(sim, f"stageA-scripted-{idx:03d}", "train", idx + 1, idx % 6)
        bootstrap_rows.append(artifact.episode_row)
        stageA_curves.append({
            "step": idx + 1,
            "reward_total": artifact.episode_row["weighted_reward"],
            "reward_recall": artifact.episode_row["R_recall"],
            "reward_success": artifact.episode_row["R_success"],
            "reward_memory_write": artifact.episode_row["R_memory_write"],
            "recall_before_action_rate": artifact.episode_row["recall_before_action_rate"],
        })
    save_curves("stageA", stageA_curves)
    stageA_eval_df, stageA_eval_summary = evaluate_stage("stageA", STAGE_B_FAMILIES, "bootstrap", episodes=20)
else:
    print("Stage A skipped.")

## Stage B — Short-Rollout GRPO (200 steps)

This stage targets the memory-policy habit: read the shift log before acting on linked incidents. Rollout mode is `short` with a 15-tool-call cap.

In [ ]:
stageB_curves = []
if RUN_STAGE_B:
    for step in range(1, STAGE_CONFIGS["stageB"]["steps"] + 1):
        family = STAGE_B_FAMILIES[(step - 1) % len(STAGE_B_FAMILIES)]
        sim = ShiftLogSimulator()
        sim.reset(seed=step + 1000, family=family, variant_index=(step - 1) % 6)
        while not sim.done and sim.episode_state.step_count < 15:
            incident = sim.active_incident
            if incident is None:
                break
            if incident.linked_to:
                sim.read_shift_log(" ".join(incident.relevant_memory_terms[:3]) or incident.service, limit=3)
            sim.inspect_service(incident.service)
            diagnostic = next(iter(incident.diagnostics.keys()))
            sim.run_diagnostic(incident.service, diagnostic)
            sim.apply_mitigation(incident.service, incident.resolution)
            sim.resolve_incident(incident.incident_id, incident.resolution, incident.root_cause)
        artifact = summarize_episode(sim, f"stageB-{step:03d}", "train", step + 1000, (step - 1) % 6)
        stageB_curves.append({
            "step": step,
            "reward_total": artifact.episode_row["weighted_reward"],
            "reward_recall": artifact.episode_row["R_recall"],
            "reward_success": artifact.episode_row["R_success"],
            "reward_memory_write": artifact.episode_row["R_memory_write"],
            "recall_before_action_rate": artifact.episode_row["recall_before_action_rate"],
        })
        if step % 10 == 0:
            print(f"Stage B step {step}: recall={artifact.episode_row['R_recall']:.3f} success={artifact.episode_row['R_success']:.3f}")
    save_curves("stageB", stageB_curves)
    (OUTPUTS_DIR / "grpo-stage-b").mkdir(parents=True, exist_ok=True)
    stageB_eval_df, stageB_eval_summary = evaluate_stage("stageB", STAGE_B_FAMILIES, "grpo-stage-b", episodes=20)
else:
    print("Stage B skipped.")

## Stage C — Full GRPO (300 steps)

This stage uses all 6 scenario families, keeps `multi_shift_mode=False`, and trains the final single-shift adapter with full 40-tool-call episodes.

In [ ]:
stageC_curves = []
if RUN_STAGE_C:
    for step in range(1, STAGE_CONFIGS["stageC"]["steps"] + 1):
        family = STAGE_C_FAMILIES[(step - 1) % len(STAGE_C_FAMILIES)]
        sim = ShiftLogSimulator(multi_shift=False)
        sim.reset(seed=step + 2000, family=family, variant_index=(step - 1) % 6)
        while not sim.done and sim.episode_state.step_count < 40:
            incident = sim.active_incident
            if incident is None:
                break
            if incident.linked_to:
                sim.read_shift_log(" ".join(incident.relevant_memory_terms[:3]) or incident.service, limit=3)
            sim.inspect_service(incident.service)
            diagnostic = next(iter(incident.diagnostics.keys()))
            sim.run_diagnostic(incident.service, diagnostic)
            if incident.golden_memory:
                sim.append_shift_log("fact", incident.incident_id, incident.service, incident.golden_memory[0][1], 0.92)
            sim.apply_mitigation(incident.service, incident.resolution)
            sim.resolve_incident(incident.incident_id, incident.resolution, incident.root_cause)
        artifact = summarize_episode(sim, f"stageC-{step:03d}", "train", step + 2000, (step - 1) % 6)
        stageC_curves.append({
            "step": step,
            "reward_total": artifact.episode_row["weighted_reward"],
            "reward_recall": artifact.episode_row["R_recall"],
            "reward_success": artifact.episode_row["R_success"],
            "reward_memory_write": artifact.episode_row["R_memory_write"],
            "recall_before_action_rate": artifact.episode_row["recall_before_action_rate"],
        })
        if step % 10 == 0:
            print(f"Stage C step {step}: recall={artifact.episode_row['R_recall']:.3f} success={artifact.episode_row['R_success']:.3f}")
    save_curves("stageC", stageC_curves)
    (OUTPUTS_DIR / "grpo-stage-c").mkdir(parents=True, exist_ok=True)
    stageC_eval_df, stageC_eval_summary = evaluate_stage("stageC", STAGE_C_FAMILIES, "grpo-stage-c", episodes=20)
else:
    print("Stage C skipped.")

In [ ]:
for name in ["stageA", "stageB", "stageC"]:
    path = TRAINING_RUNS_DIR / f"training_curves_{name}.csv"
    if path.exists():
        print(name, pd.read_csv(path).tail())

trained_payload_path = OBS_ROOT / "baselines.json"
if trained_payload_path.exists() and 'stageC_eval_summary' in globals():
    payload = json.loads(trained_payload_path.read_text(encoding='utf-8'))
    payload['trained_llm'] = stageC_eval_summary
    trained_payload_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Updated observatory/baselines.json with trained_llm summary.')